# 17. Train pooled calibration predictor with paired +δ / -δ priors

This notebook is the direct follow-up to notebook 16 after the stochastic-small-set diagnostic showed collapse toward the zero/mean correction.

The architecture is intentionally unchanged. The experimental change is the supervision:

- each base motion window is augmented once with shared frame randomization;
- one calibration-prior perturbation `δ` is sampled for each calibration and batch item;
- two branches are constructed from the same measurements: `+δ` and `-δ`;
- ordinary normalized correction regression is applied to both branches;
- an additional pair-difference loss forces the predicted `(+δ) - (-δ)` response to match the true pair difference.

The clean starting distribution is:

- frame randomization: **enabled**;
- true-calibration augmentation: **disabled**;
- measurement noise: **disabled**;
- sampling-rate augmentation: **disabled**;
- calibration events: **disabled**;
- prior rotation bound: **5 deg**;
- prior translation bound: **2 cm**;
- prior time-offset bound: **10 ms**.

The notebook expects `apply_prior_perturbation(...)` to exist in `obscalib.augmentations.prior_perturbation`. The required backward-compatible source change is described alongside this notebook.


## 1. Imports and repository root


In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "obscalib").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the nn-obs-calib repository root containing src/obscalib.")

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, TrueCalibrationAugmentationConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.augmentations.prior_perturbation import CalibrationPriorPerturber, apply_prior_perturbation
from obscalib.calibration.state import CalibrationState
from obscalib.calibration.update import CalibrationUpdater
from obscalib.config import CalibrationPoolerConfig, MLPConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import WindowBatch
from obscalib.geometry.lie import se3_log
from obscalib.geometry.processing import GeometryProcessor
from obscalib.models.pooled_calibration_model import PooledCalibrationModel
from obscalib.pipeline.window_step import WindowStep
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint
from obscalib.training.metrics_logger import MetricsLogger

print("Repository root:", ROOT)
print("PyTorch:", torch.__version__)


## 2. Experiment parameters


In [ ]:
RUN_NAME = "kaist_pooled_calibration_paired_002"

RANDOM_SEED = 12345
TRAIN_PAIR_SEED = 23456
VALIDATION_PAIR_SEED = 34567
TEST_PAIR_SEED = 45678
VISUALIZATION_PAIR_SEED = 56789

DEVICE_NAME = "cuda"
DEVICE = torch.device("cuda" if DEVICE_NAME == "auto" and torch.cuda.is_available() else DEVICE_NAME)
MODEL_DTYPE = torch.float32

# Stable routing order for this rig. IDs are tuple positions only.
CALIBRATION_KEYS = ("imu", "lidar")
CALIBRATION_CONTEXT_DIM = 7

WINDOW_DURATION_S = 5.0
WINDOW_STRIDE_S = 5.0
MAX_SAMPLES_PER_SENSOR = 700

BATCH_SIZE = 16
NUM_WORKERS = 0
DROP_LAST_TRAIN = True

NUM_EPOCHS = 200
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0
CHECKPOINT_EVERY_EPOCHS = 3
PRINT_EVERY_BATCHES = 500
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE_EPOCHS = 10

MEASUREMENT_DIM = 6
TOKEN_INPUT_DIM = MEASUREMENT_DIM + 2
D_MODEL = 128
N_HEADS = 8
NUM_TRANSFORMER_LAYERS = 12
DIM_FEEDFORWARD = 512
TRANSFORMER_DROPOUT = 0.1
NUM_SUMMARY_TOKENS = 1

POOLER_N_HEADS = 8
POOLER_DROPOUT = 0.1
MLP_HIDDEN_DIMS = (256, 128)
SHARED_FEATURE_DIM = 128
MLP_DROPOUT = 0.1
HEAD_HIDDEN_DIMS = (128, 64)
HEAD_DROPOUT = 0.1

CORRECTION_LOSS_TYPE = "mse"
ROTATION_LOSS_SCALE_RAD = np.deg2rad(5.0)
TRANSLATION_LOSS_SCALE_M = 0.05
TIME_OFFSET_LOSS_SCALE_S = 0.015

ROTATION_LOSS_WEIGHT = 2.0
TRANSLATION_LOSS_WEIGHT = 1.0
TIME_OFFSET_LOSS_WEIGHT = 1.0

# Pair difference is the new supervision term. The normal regression loss remains present on both branches.
PAIR_LOSS_WEIGHT = 1.0

# Keep event heads in the architecture but do not train them in this clean paired experiment.
CHANGE_EVENT_LOSS_WEIGHT = 0.0
CHANGE_TIME_LOSS_WEIGHT = 0.0

PAIR_ROTATION_MAX_RAD = np.deg2rad(5.0)
PAIR_TRANSLATION_MAX_M = 0.02
PAIR_TIME_OFFSET_MAX_S = 0.01

RESUME_CHECKPOINT = None
# RESUME_CHECKPOINT = ROOT / "runs" / RUN_NAME / "last.pt"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Device:", DEVICE)
print("Calibration order:", CALIBRATION_KEYS)
print("Correction loss:", CORRECTION_LOSS_TYPE)
print("Pair loss weight:", PAIR_LOSS_WEIGHT)
print("Pair prior bounds: rotation=5 deg, translation=2 cm, time offset=10 ms")


## 3. KAIST split, calibration, datasets, and loaders


In [ ]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)


In [ ]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT)
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

print(train_dataset.summary())
print()
print(val_dataset.summary())
print()
print(test_dataset.summary())


In [ ]:
train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=DROP_LAST_TRAIN, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
test_loader = make_kaist_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))


## 4. Shared context augmentation and paired prior sampler


In [ ]:
# Shared context augmentation is applied ONCE before the pair is split.
# Therefore +delta and -delta see exactly the same physical motion, frame randomization, targets, and streams.
#
# For this clean experiment:
#   frame randomization: ON
#   true-calibration augmentation: OFF
#   prior perturbation inside the generic pipeline: OFF
#   measurement noise: OFF
#   sampling-rate augmentation: OFF
#   calibration events: OFF
#
# The paired prior itself is sampled separately below.

context_augmentation_config = AugmentationConfig(
    sampling_rate=SamplingRateAugmentationConfig(enabled=False),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    true_calibration=TrueCalibrationAugmentationConfig(
        enabled=True,
        probability=1.0,
        rotation_probability=1.0,
        time_offset_probability=0.0,
        rotation=PerturbationMagnitudeConfig(
            maximum=np.deg2rad(180.0),
        ),
        time_offset=None,
    ),
    prior_perturbation=PriorPerturbationConfig(enabled=False),
    calibration_event=CalibrationEventConfig(enabled=False),
    noise=NoiseAugmentationConfig(enabled=False),
)

pair_prior_config = PriorPerturbationConfig(
    enabled=True,
    probability=1.0,
    rotation=PerturbationMagnitudeConfig(maximum=PAIR_ROTATION_MAX_RAD),
    translation=PerturbationMagnitudeConfig(maximum=PAIR_TRANSLATION_MAX_M),
    time_offset=PerturbationMagnitudeConfig(maximum=PAIR_TIME_OFFSET_MAX_S),
)

context_augmentation_pipeline = AugmentationPipeline(context_augmentation_config)
pair_prior_perturber = CalibrationPriorPerturber(pair_prior_config)

print("Shared context augmentation:")
print(context_augmentation_config)
print()
print("Paired prior perturbation:")
print(pair_prior_config)


## 5. Pooled model and complete window pipeline


In [ ]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, num_summary_tokens=NUM_SUMMARY_TOKENS, activation="gelu"),
    mlp=MLPConfig(hidden_dims=MLP_HIDDEN_DIMS, output_dim=SHARED_FEATURE_DIM, activation="gelu", dropout=MLP_DROPOUT),
    pooler=CalibrationPoolerConfig(
        n_heads=POOLER_N_HEADS,
        dropout=POOLER_DROPOUT,
        calibration_context_dim=CALIBRATION_CONTEXT_DIM,
    ),

    head=PooledCalibrationHeadConfig(
        calibration_context_dim=0,
        hidden_dims=HEAD_HIDDEN_DIMS,
        activation="gelu",
        dropout=HEAD_DROPOUT,
    ),
)

geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
model = PooledCalibrationModel(model_config).to(device=DEVICE, dtype=MODEL_DTYPE)
calibration_updater = CalibrationUpdater()
window_step = WindowStep(geometry_processor=geometry_processor, tokenizer=tokenizer, model=model, calibration_updater=calibration_updater)

if not getattr(model, "uses_calibration_context", False):
    raise RuntimeError("This experiment requires the pooled model to use numeric calibration context.")

print(model)
print()
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))
print("Pooler calibration context dim:", model.pooler.calibration_context_dim)
print("Head calibration context dim:", model.head.config.calibration_context_dim)


## 6. Paired-window construction helpers


In [ ]:
def make_augmentation_generator(seed: int):
    return torch.Generator(device=DEVICE.type).manual_seed(seed)


def clone_window_with_calibration(window: WindowBatch, calibration):
    return WindowBatch(
        streams=dict(window.streams),
        current_calibration=calibration,
        metadata=dict(window.metadata),
        targets=None if window.targets is None else dict(window.targets),
        dataset_indices=window.dataset_indices,
    )


def make_paired_windows(base_window: WindowBatch, *, generator: torch.Generator):
    # Apply all shared physical/context augmentation once.
    context_result = context_augmentation_pipeline(base_window, generator=generator)
    context_window = context_result.augmented_window

    # Because generic prior perturbation and calibration events are disabled, this is the true calibration
    # associated with the shared measurements and targets.
    true_calibration = context_window.current_calibration

    # Reuse the existing sampler only to draw one delta per calibration and batch item.
    _, delta_xi_by_key, delta_tau_by_key = pair_prior_perturber(true_calibration, generator=generator)

    positive_prior = apply_prior_perturbation(true_calibration, delta_xi_by_key, delta_tau_by_key)
    negative_prior = apply_prior_perturbation(
        true_calibration,
        {key: -value for key, value in delta_xi_by_key.items()},
        {key: -value for key, value in delta_tau_by_key.items()},
    )

    positive_window = clone_window_with_calibration(context_window, positive_prior)
    negative_window = clone_window_with_calibration(context_window, negative_prior)

    return positive_window, negative_window, delta_xi_by_key, delta_tau_by_key


def correction_target(window: WindowBatch, calibration_key: str):
    if window.targets is None:
        raise ValueError("Window has no calibration targets.")

    target = window.targets[calibration_key]
    current_state = window.current_calibration[calibration_key]

    if target.next_transform is None or target.next_time_offset is None:
        raise ValueError(f"Calibration target {calibration_key!r} does not contain the final calibration state.")

    target_delta_transform = target.next_transform @ torch.linalg.inv(current_state.transform)
    target_delta_xi = se3_log(target_delta_transform)
    target_delta_tau = target.next_time_offset - current_state.time_offset
    return target_delta_xi, target_delta_tau


## 7. Pair-construction semantic test


In [ ]:
semantic_base_window = next(iter(train_loader))
semantic_base_window = window_batch_to(semantic_base_window, DEVICE, dtype=MODEL_DTYPE)
semantic_generator = make_augmentation_generator(TRAIN_PAIR_SEED + 1)
semantic_positive_window, semantic_negative_window, semantic_delta_xi, semantic_delta_tau = make_paired_windows(semantic_base_window, generator=semantic_generator)

for stream_key in semantic_positive_window.streams:
    torch.testing.assert_close(semantic_positive_window.streams[stream_key].values, semantic_negative_window.streams[stream_key].values)
    torch.testing.assert_close(semantic_positive_window.streams[stream_key].timestamps, semantic_negative_window.streams[stream_key].timestamps)

for calibration_key in CALIBRATION_KEYS:
    positive_target_xi, positive_target_tau = correction_target(semantic_positive_window, calibration_key)
    negative_target_xi, negative_target_tau = correction_target(semantic_negative_window, calibration_key)

    # For T_prior = Exp(+delta) @ T_true, the desired correction is -delta.
    torch.testing.assert_close(positive_target_xi, -semantic_delta_xi[calibration_key], atol=1e-5, rtol=1e-4)
    torch.testing.assert_close(negative_target_xi, semantic_delta_xi[calibration_key], atol=1e-5, rtol=1e-4)
    torch.testing.assert_close(positive_target_tau, -semantic_delta_tau[calibration_key], atol=1e-7, rtol=1e-6)
    torch.testing.assert_close(negative_target_tau, semantic_delta_tau[calibration_key], atol=1e-7, rtol=1e-6)

    torch.testing.assert_close(positive_target_xi + negative_target_xi, torch.zeros_like(positive_target_xi), atol=1e-5, rtol=1e-4)
    torch.testing.assert_close(positive_target_tau + negative_target_tau, torch.zeros_like(positive_target_tau), atol=1e-7, rtol=1e-6)

print("PAIR CONSTRUCTION CHECK: PASS")
print("The +delta and -delta branches share identical measurements and have opposite correction targets.")


## 8. Routing and pooled-output sanity check


In [ ]:
def run_pooled_sanity(window):
    with torch.inference_mode():
        result = window_step(window)
        tokens = result.tokens
        tokens.validate()

        if tokens.calibration_ids is None:
            raise RuntimeError("Pooled calibration requires calibration_ids.")

        if tuple(tokens.calibration_keys) != CALIBRATION_KEYS:
            raise RuntimeError(
                f"Tokenizer calibration order {tuple(tokens.calibration_keys)} "
                f"does not match requested {CALIBRATION_KEYS}."
            )

        if tuple(result.model_output.predictions) != CALIBRATION_KEYS:
            raise RuntimeError(
                f"Prediction key order {tuple(result.model_output.predictions)} "
                f"does not match {CALIBRATION_KEYS}."
            )

        summary = {
            "token_shape": tuple(tokens.x.shape),
            "shared_feature_shape": (
                None
                if result.model_output.shared_features is None
                else tuple(result.model_output.shared_features.shape)
            ),
        }

        # Move only the tiny outputs needed for comparison to CPU.
        predictions = {
            calibration_key: {
                "delta_xi": prediction.delta_xi.detach().cpu(),
                "delta_tau": prediction.delta_tau.detach().cpu(),
            }
            for calibration_key, prediction in result.model_output.predictions.items()
        }

    return summary, predictions


previous_training_mode = model.training
model.eval()

positive_summary, positive_predictions = run_pooled_sanity(
    semantic_positive_window
)

negative_summary, negative_predictions = run_pooled_sanity(
    semantic_negative_window
)

model.train(previous_training_mode)


for result_name, summary in (
    ("positive", positive_summary),
    ("negative", negative_summary),
):
    print(result_name)
    print("  token shape:", summary["token_shape"])
    print(
        "  pooled/shared feature shape:",
        summary["shared_feature_shape"],
    )


print()

for calibration_key in CALIBRATION_KEYS:
    positive_prediction = positive_predictions[calibration_key]
    negative_prediction = negative_predictions[calibration_key]

    print(calibration_key)
    print(
        "  delta_xi:",
        tuple(positive_prediction["delta_xi"].shape),
    )
    print(
        "  delta_tau:",
        tuple(positive_prediction["delta_tau"].shape),
    )
    print(
        "  initial prediction separation xi:",
        float(
            torch.linalg.vector_norm(
                positive_prediction["delta_xi"]
                - negative_prediction["delta_xi"],
                dim=-1,
            ).mean()
        ),
    )


# These are CPU-only now.
del positive_predictions
del negative_predictions
del positive_summary
del negative_summary
del positive_prediction
del negative_prediction

torch.cuda.empty_cache()

## 9. Regression and pair-difference losses


In [ ]:
def correction_regression_loss(prediction, target):
    if CORRECTION_LOSS_TYPE == "mse":
        return F.mse_loss(prediction, target)
    if CORRECTION_LOSS_TYPE == "smooth_l1":
        return F.smooth_l1_loss(prediction, target, beta=1.0)
    if CORRECTION_LOSS_TYPE == "l1":
        return F.l1_loss(prediction, target)
    raise ValueError(f"Unknown CORRECTION_LOSS_TYPE={CORRECTION_LOSS_TYPE!r}.")


def compute_training_loss(window, model_output):
    total_loss = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
    metrics = {}
    num_calibrations = 0

    for calibration_key, prediction in model_output.predictions.items():
        target_delta_xi, target_delta_tau = correction_target(window, calibration_key)

        rotation_loss = correction_regression_loss(prediction.delta_xi[:, :3] / ROTATION_LOSS_SCALE_RAD, target_delta_xi[:, :3] / ROTATION_LOSS_SCALE_RAD)
        translation_loss = correction_regression_loss(prediction.delta_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M, target_delta_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M)
        time_offset_loss = correction_regression_loss(prediction.delta_tau / TIME_OFFSET_LOSS_SCALE_S, target_delta_tau / TIME_OFFSET_LOSS_SCALE_S)

        calibration_loss = ROTATION_LOSS_WEIGHT * rotation_loss + TRANSLATION_LOSS_WEIGHT * translation_loss + TIME_OFFSET_LOSS_WEIGHT * time_offset_loss
        total_loss = total_loss + calibration_loss
        num_calibrations += 1

        rotation_error_rad = torch.linalg.vector_norm(prediction.delta_xi[:, :3] - target_delta_xi[:, :3], dim=-1)
        translation_error_m = torch.linalg.vector_norm(prediction.delta_xi[:, 3:] - target_delta_xi[:, 3:], dim=-1)
        time_offset_error_s = torch.abs(prediction.delta_tau[:, 0] - target_delta_tau[:, 0])

        metrics[f"{calibration_key}/rotation_error_deg"] = torch.rad2deg(rotation_error_rad).mean().item()
        metrics[f"{calibration_key}/translation_error_cm"] = (100.0 * translation_error_m).mean().item()
        metrics[f"{calibration_key}/time_offset_error_ms"] = (1000.0 * time_offset_error_s).mean().item()
        metrics[f"{calibration_key}/loss_rotation"] = rotation_loss.detach().item()
        metrics[f"{calibration_key}/loss_translation"] = translation_loss.detach().item()
        metrics[f"{calibration_key}/loss_time_offset"] = time_offset_loss.detach().item()

    if num_calibrations == 0:
        raise ValueError("Model produced no calibration predictions.")

    total_loss = total_loss / num_calibrations
    metrics["loss_total"] = total_loss.detach().item()
    return total_loss, metrics


def compute_pair_difference_loss(positive_window, positive_output, negative_window, negative_output):
    total_pair_loss = torch.zeros((), dtype=MODEL_DTYPE, device=DEVICE)
    metrics = {}
    num_calibrations = 0

    for calibration_key in CALIBRATION_KEYS:
        positive_prediction = positive_output.predictions[calibration_key]
        negative_prediction = negative_output.predictions[calibration_key]

        positive_target_xi, positive_target_tau = correction_target(positive_window, calibration_key)
        negative_target_xi, negative_target_tau = correction_target(negative_window, calibration_key)

        predicted_difference_xi = positive_prediction.delta_xi - negative_prediction.delta_xi
        target_difference_xi = positive_target_xi - negative_target_xi
        predicted_difference_tau = positive_prediction.delta_tau - negative_prediction.delta_tau
        target_difference_tau = positive_target_tau - negative_target_tau

        pair_rotation_loss = correction_regression_loss(predicted_difference_xi[:, :3] / ROTATION_LOSS_SCALE_RAD, target_difference_xi[:, :3] / ROTATION_LOSS_SCALE_RAD)
        pair_translation_loss = correction_regression_loss(predicted_difference_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M, target_difference_xi[:, 3:] / TRANSLATION_LOSS_SCALE_M)
        pair_time_offset_loss = correction_regression_loss(predicted_difference_tau / TIME_OFFSET_LOSS_SCALE_S, target_difference_tau / TIME_OFFSET_LOSS_SCALE_S)

        calibration_pair_loss = ROTATION_LOSS_WEIGHT * pair_rotation_loss + TRANSLATION_LOSS_WEIGHT * pair_translation_loss + TIME_OFFSET_LOSS_WEIGHT * pair_time_offset_loss
        total_pair_loss = total_pair_loss + calibration_pair_loss
        num_calibrations += 1

        metrics[f"{calibration_key}/pair_loss_rotation"] = pair_rotation_loss.detach().item()
        metrics[f"{calibration_key}/pair_loss_translation"] = pair_translation_loss.detach().item()
        metrics[f"{calibration_key}/pair_loss_time_offset"] = pair_time_offset_loss.detach().item()

    total_pair_loss = total_pair_loss / num_calibrations
    metrics["loss_pair"] = total_pair_loss.detach().item()
    return total_pair_loss, metrics


def compute_paired_objective(positive_window, positive_output, negative_window, negative_output):
    positive_loss, positive_metrics = compute_training_loss(positive_window, positive_output)
    negative_loss, negative_metrics = compute_training_loss(negative_window, negative_output)
    pair_loss, pair_metrics = compute_pair_difference_loss(positive_window, positive_output, negative_window, negative_output)

    regular_loss = 0.5 * (positive_loss + negative_loss)
    total_loss = regular_loss + PAIR_LOSS_WEIGHT * pair_loss

    metrics = {
        "loss_total": total_loss.detach().item(),
        "loss_regular": regular_loss.detach().item(),
        "loss_pair": pair_loss.detach().item(),
    }

    for key in positive_metrics:
        if key == "loss_total":
            continue
        metrics[f"positive/{key}"] = positive_metrics[key]
        metrics[f"negative/{key}"] = negative_metrics[key]
        metrics[f"mean/{key}"] = 0.5 * (positive_metrics[key] + negative_metrics[key])

    metrics.update({f"pair/{key}": value for key, value in pair_metrics.items() if key != "loss_pair"})
    return total_loss, metrics


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_REDUCTION_FACTOR, patience=LR_PATIENCE_EPOCHS)


## 10. Metric helpers


In [ ]:
def accumulate_metrics(metric_sums, metrics, batch_size):
    for key, value in metrics.items():
        metric_sums[key] = metric_sums.get(key, 0.0) + float(value) * batch_size


def normalize_metrics(metric_sums, num_samples):
    if num_samples <= 0:
        raise ValueError("num_samples must be positive.")
    return {key: value / num_samples for key, value in metric_sums.items()}


## 11. One-batch paired gradient smoke test


In [ ]:
from obscalib.calibration.context import calibration_state_to_context

context_test_base_window = next(iter(train_loader))
context_test_base_window = window_batch_to(
    context_test_base_window,
    DEVICE,
    dtype=MODEL_DTYPE,
)

context_test_generator = make_augmentation_generator(
    TRAIN_PAIR_SEED + 123
)

(
    context_test_positive_window,
    context_test_negative_window,
    _,
    _,
) = make_paired_windows(
    context_test_base_window,
    generator=context_test_generator,
)

positive_context = {
    calibration_key: calibration_state_to_context(state)
    for calibration_key, state
    in context_test_positive_window.current_calibration.items()
}

negative_context = {
    calibration_key: calibration_state_to_context(state)
    for calibration_key, state
    in context_test_negative_window.current_calibration.items()
}

model.eval()

with torch.inference_mode():
    # Generate tokens once.
    positive_step = window_step(
        context_test_positive_window
    )

    fixed_tokens = positive_step.tokens

    # Same exact measurement tokens, different numeric priors.
    output_positive_context = model(
        fixed_tokens,
        positive_context,
    )

    output_negative_context = model(
        fixed_tokens,
        negative_context,
    )

print("=" * 90)
print("CALIBRATION CONTEXT PATH CHECK")
print("=" * 90)

for calibration_key in CALIBRATION_KEYS:
    positive_prediction = (
        output_positive_context
        .predictions[calibration_key]
    )

    negative_prediction = (
        output_negative_context
        .predictions[calibration_key]
    )

    xi_separation = torch.linalg.vector_norm(
        positive_prediction.delta_xi
        - negative_prediction.delta_xi,
        dim=-1,
    ).mean()

    tau_separation = torch.abs(
        positive_prediction.delta_tau
        - negative_prediction.delta_tau
    ).mean()

    print(calibration_key)
    print(
        "  xi separation:",
        float(xi_separation.cpu()),
    )
    print(
        "  tau separation:",
        float(tau_separation.cpu()),
    )

del positive_step
del output_positive_context
del output_negative_context
del fixed_tokens
del context_test_base_window
del context_test_positive_window
del context_test_negative_window

torch.cuda.empty_cache()

In [ ]:
# Paired gradient smoke test.
#
# Use a deliberately tiny batch. This test only verifies that the paired
# objective produces finite, nonzero gradients through the model. It does
# not need to reproduce the memory footprint of full training.

SMOKE_BATCH_SIZE = 2

previous_training_mode = model.training

try:
    torch.cuda.empty_cache()

    ##################################################
    # Tiny smoke batch
    ##################################################

    smoke_indices = np.linspace(
        0,
        len(train_dataset) - 1,
        SMOKE_BATCH_SIZE,
        dtype=int,
    )

    smoke_base_window = collate_windows(
        [
            train_dataset[int(index)]
            for index in smoke_indices
        ]
    )

    smoke_base_window = window_batch_to(
        smoke_base_window,
        DEVICE,
        dtype=MODEL_DTYPE,
    )

    smoke_generator = make_augmentation_generator(
        TRAIN_PAIR_SEED + 2
    )

    (
        smoke_positive_window,
        smoke_negative_window,
        _,
        _,
    ) = make_paired_windows(
        smoke_base_window,
        generator=smoke_generator,
    )

    ##################################################
    # Forward + backward
    ##################################################

    model.train()
    optimizer.zero_grad(set_to_none=True)

    smoke_positive_result = window_step(
        smoke_positive_window
    )

    smoke_negative_result = window_step(
        smoke_negative_window
    )

    smoke_loss, smoke_metrics = compute_paired_objective(
        smoke_positive_window,
        smoke_positive_result.model_output,
        smoke_negative_window,
        smoke_negative_result.model_output,
    )

    smoke_loss.backward()

    ##################################################
    # Gradient diagnostic
    ##################################################

    gradient_norm_squared = 0.0
    num_parameters_with_gradient = 0

    for parameter in model.parameters():
        if parameter.grad is None:
            continue

        parameter_gradient_norm = float(
            torch.linalg.vector_norm(
                parameter.grad.detach()
            ).cpu()
        )

        gradient_norm_squared += (
            parameter_gradient_norm ** 2
        )

        num_parameters_with_gradient += 1

    gradient_norm = (
        gradient_norm_squared ** 0.5
    )

    smoke_loss_value = float(
        smoke_loss.detach().cpu()
    )

    print(
        "Paired smoke loss:",
        smoke_loss_value,
    )

    print(
        "  regular:",
        smoke_metrics["loss_regular"],
    )

    print(
        "  pair:",
        smoke_metrics["loss_pair"],
    )

    print(
        "Gradient norm:",
        gradient_norm,
    )

    print(
        "Parameters with gradients:",
        num_parameters_with_gradient,
    )

    if not np.isfinite(smoke_loss_value):
        raise RuntimeError(
            "Paired smoke loss is not finite."
        )

    if (
        not np.isfinite(gradient_norm)
        or gradient_norm <= 0.0
    ):
        raise RuntimeError(
            "Pooled model did not receive a finite "
            "nonzero paired-training gradient."
        )

    print(
        "PAIRED GRADIENT SMOKE CHECK: PASS"
    )

finally:
    ##################################################
    # Remove gradients and all large graph references
    ##################################################

    optimizer.zero_grad(
        set_to_none=True
    )

    model.train(
        previous_training_mode
    )

    for variable_name in (
        "smoke_positive_result",
        "smoke_negative_result",
        "smoke_loss",
        "smoke_metrics",
        "smoke_positive_window",
        "smoke_negative_window",
        "smoke_base_window",
        "smoke_generator",
    ):
        if variable_name in globals():
            del globals()[variable_name]

    torch.cuda.empty_cache()

## 12. Fixed paired small-batch overfit test

This is the direct counterpart of the successful fixed overfit test from notebook 16. It verifies that the new pair objective itself is trainable. Both branches and their perturbations are frozen here.


In [ ]:
FIXED_PAIR_OVERFIT_NUM_WINDOWS = min(BATCH_SIZE, 16)
FIXED_PAIR_OVERFIT_NUM_STEPS = 400
FIXED_PAIR_OVERFIT_PRINT_EVERY = 25
FIXED_PAIR_OVERFIT_SEED = TRAIN_PAIR_SEED + 100000

fixed_pair_indices = np.linspace(0, len(train_dataset) - 1, FIXED_PAIR_OVERFIT_NUM_WINDOWS, dtype=int)
fixed_pair_base_window = collate_windows([train_dataset[int(index)] for index in fixed_pair_indices])
fixed_pair_base_window = window_batch_to(fixed_pair_base_window, DEVICE, dtype=MODEL_DTYPE)

fixed_pair_generator = make_augmentation_generator(FIXED_PAIR_OVERFIT_SEED)
fixed_pair_positive_window, fixed_pair_negative_window, _, _ = make_paired_windows(fixed_pair_base_window, generator=fixed_pair_generator)

fixed_pair_model_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
fixed_pair_previous_training_mode = model.training
fixed_pair_optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
fixed_pair_history = []

print("=" * 110)
print("FIXED PAIRED SMALL-BATCH OVERFIT DIAGNOSTIC")
print("=" * 110)
torch.cuda.empty_cache()
model.train()

for step in range(FIXED_PAIR_OVERFIT_NUM_STEPS + 1):
    fixed_pair_optimizer.zero_grad(set_to_none=True)

    positive_result = window_step(fixed_pair_positive_window)
    negative_result = window_step(fixed_pair_negative_window)
    loss, metrics = compute_paired_objective(fixed_pair_positive_window, positive_result.model_output, fixed_pair_negative_window, negative_result.model_output)
    fixed_pair_history.append(float(loss.detach()))

    if step == 0 or step % FIXED_PAIR_OVERFIT_PRINT_EVERY == 0 or step == FIXED_PAIR_OVERFIT_NUM_STEPS:
        print(f"step {step:4d}/{FIXED_PAIR_OVERFIT_NUM_STEPS} total={loss.item():9.6f} regular={metrics['loss_regular']:9.6f} pair={metrics['loss_pair']:9.6f}")

    if step == FIXED_PAIR_OVERFIT_NUM_STEPS:
        break

    loss.backward()
    if GRADIENT_CLIP_NORM is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
    fixed_pair_optimizer.step()

fixed_pair_fraction_remaining = fixed_pair_history[-1] / fixed_pair_history[0]

print()
print(f"Initial loss: {fixed_pair_history[0]:.6f}")
print(f"Final loss:   {fixed_pair_history[-1]:.6f}")
print(f"Remaining:    {100.0 * fixed_pair_fraction_remaining:.2f}% of initial loss")
print("FIXED PAIRED OVERFIT CHECK:", "PASS" if fixed_pair_fraction_remaining < 0.10 else "FAIL/WARNING")

plt.figure(figsize=(10, 4))
plt.plot(fixed_pair_history)
plt.yscale("log")
plt.xlabel("optimization step")
plt.ylabel("paired objective")
plt.title("Fixed paired small-batch overfit diagnostic")
plt.grid(True)
plt.show()

model.load_state_dict(fixed_pair_model_state)
model.train(fixed_pair_previous_training_mode)
model.zero_grad(set_to_none=True)

del fixed_pair_optimizer
del fixed_pair_model_state
torch.cuda.empty_cache()

## 13. Stochastic paired small-set learnability gate

The physical windows stay fixed, but shared frame randomization and the paired prior perturbation are resampled every step. The model is evaluated on one fixed unseen paired draw. This is the decisive small test before the full run.

The full 200-epoch run is enabled automatically only if the fixed-evaluation paired objective falls below 50% of its initial value.


In [ ]:
STOCHASTIC_PAIR_NUM_WINDOWS = min(BATCH_SIZE, 16)
STOCHASTIC_PAIR_NUM_STEPS = 400
STOCHASTIC_PAIR_PRINT_EVERY = 25
STOCHASTIC_PAIR_TRAIN_SEED = TRAIN_PAIR_SEED + 200000
STOCHASTIC_PAIR_EVAL_SEED = TRAIN_PAIR_SEED + 300000

stochastic_pair_indices = np.linspace(0, len(train_dataset) - 1, STOCHASTIC_PAIR_NUM_WINDOWS, dtype=int)
stochastic_pair_base_window = collate_windows([train_dataset[int(index)] for index in stochastic_pair_indices])
stochastic_pair_base_window = window_batch_to(stochastic_pair_base_window, DEVICE, dtype=MODEL_DTYPE)

stochastic_pair_eval_generator = make_augmentation_generator(STOCHASTIC_PAIR_EVAL_SEED)
stochastic_pair_eval_positive_window, stochastic_pair_eval_negative_window, _, _ = make_paired_windows(stochastic_pair_base_window, generator=stochastic_pair_eval_generator)
stochastic_pair_train_generator = make_augmentation_generator(STOCHASTIC_PAIR_TRAIN_SEED)

stochastic_pair_model_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
stochastic_pair_previous_training_mode = model.training
stochastic_pair_optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

stochastic_pair_train_history = []
stochastic_pair_eval_history = []
stochastic_pair_eval_regular_history = []
stochastic_pair_eval_difference_history = []
stochastic_pair_eval_steps = []

torch.cuda.empty_cache()
def evaluate_fixed_stochastic_pair():
    model.eval()
    with torch.no_grad():
        positive_result = window_step(stochastic_pair_eval_positive_window)
        negative_result = window_step(stochastic_pair_eval_negative_window)
        loss, metrics = compute_paired_objective(stochastic_pair_eval_positive_window, positive_result.model_output, stochastic_pair_eval_negative_window, negative_result.model_output)
    return float(loss), metrics


initial_stochastic_pair_eval_loss, initial_stochastic_pair_eval_metrics = evaluate_fixed_stochastic_pair()

print("=" * 110)
print("STOCHASTIC PAIRED SMALL-SET LEARNABILITY DIAGNOSTIC")
print("=" * 110)
print("fixed physical windows:", STOCHASTIC_PAIR_NUM_WINDOWS)
print("optimization steps:", STOCHASTIC_PAIR_NUM_STEPS)
print("fresh shared frame randomization and fresh +/- prior pair sampled at every training step")
print(f"initial fixed-eval total={initial_stochastic_pair_eval_loss:.6f} regular={initial_stochastic_pair_eval_metrics['loss_regular']:.6f} pair={initial_stochastic_pair_eval_metrics['loss_pair']:.6f}")
print()

for step in range(1, STOCHASTIC_PAIR_NUM_STEPS + 1):
    model.train()

    positive_window, negative_window, _, _ = make_paired_windows(stochastic_pair_base_window, generator=stochastic_pair_train_generator)

    stochastic_pair_optimizer.zero_grad(set_to_none=True)
    positive_result = window_step(positive_window)
    negative_result = window_step(negative_window)
    loss, metrics = compute_paired_objective(positive_window, positive_result.model_output, negative_window, negative_result.model_output)

    loss.backward()
    if GRADIENT_CLIP_NORM is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
    stochastic_pair_optimizer.step()

    stochastic_pair_train_history.append(float(loss.detach()))

    if step == 1 or step % STOCHASTIC_PAIR_PRINT_EVERY == 0 or step == STOCHASTIC_PAIR_NUM_STEPS:
        eval_loss, eval_metrics = evaluate_fixed_stochastic_pair()
        stochastic_pair_eval_steps.append(step)
        stochastic_pair_eval_history.append(eval_loss)
        stochastic_pair_eval_regular_history.append(eval_metrics["loss_regular"])
        stochastic_pair_eval_difference_history.append(eval_metrics["loss_pair"])

        print(
            f"step {step:4d}/{STOCHASTIC_PAIR_NUM_STEPS} "
            f"train={loss.item():9.6f} "
            f"eval_total={eval_loss:9.6f} "
            f"eval_regular={eval_metrics['loss_regular']:9.6f} "
            f"eval_pair={eval_metrics['loss_pair']:9.6f}"
        )

initial_pair_loss = (initial_stochastic_pair_eval_metrics["loss_pair"])

final_pair_loss = (stochastic_pair_eval_difference_history[-1])

pair_fraction_remaining = (final_pair_loss / initial_pair_loss)

initial_regular_loss = (initial_stochastic_pair_eval_metrics["loss_regular"])

final_regular_loss = (stochastic_pair_eval_regular_history[-1])

regular_fraction_remaining = (final_regular_loss / initial_regular_loss)

PAIR_GATE_THRESHOLD = 0.50

PAIRED_STOCHASTIC_GATE_PASSED = (pair_fraction_remaining < PAIR_GATE_THRESHOLD)

print()
print(f"Initial regular loss: {initial_regular_loss:.6f}")
print(f"Final regular loss:   {final_regular_loss:.6f}")
print(f"Regular remaining:    {100.0 * regular_fraction_remaining:.2f}%")

print()
print(f"Initial pair loss: {initial_pair_loss:.6f}")
print(f"Final pair loss:   {final_pair_loss:.6f}")
print(f"Pair remaining:    {100.0 * pair_fraction_remaining:.2f}%")

print()

print("STOCHASTIC PAIRED LEARNABILITY CHECK:",( "PASS" if PAIRED_STOCHASTIC_GATE_PASSED else "FAIL/WARNING"),)

plt.figure(figsize=(11, 4))
plt.plot(stochastic_pair_train_history, alpha=0.55, label="fresh-pair training loss")
plt.plot(stochastic_pair_eval_steps, stochastic_pair_eval_history, marker="o", label="fixed pair total")
plt.plot(stochastic_pair_eval_steps, stochastic_pair_eval_regular_history, marker="o", label="fixed pair regular")
plt.plot(stochastic_pair_eval_steps, stochastic_pair_eval_difference_history, marker="o", label="fixed pair difference")
plt.xlabel("optimization step")
plt.ylabel("loss")
plt.title("Stochastic paired small-set learnability diagnostic")
plt.grid(True)
plt.legend()
plt.show()

model.load_state_dict(stochastic_pair_model_state)
model.train(stochastic_pair_previous_training_mode)
model.zero_grad(set_to_none=True)

del stochastic_pair_optimizer
del stochastic_pair_model_state

RUN_FULL_TRAINING = PAIRED_STOCHASTIC_GATE_PASSED
print("RUN_FULL_TRAINING =", RUN_FULL_TRAINING)
if not RUN_FULL_TRAINING:
    print("Full 200-epoch run is gated off because the paired stochastic small-set diagnostic did not pass.")
torch.cuda.empty_cache()    

In [ ]:
from obscalib.augmentations.prior_perturbation import apply_prior_perturbation

TOKEN_SENSITIVITY_NUM_WINDOWS = min(BATCH_SIZE, 8)
TOKEN_SENSITIVITY_ROTATION_RAD = np.deg2rad(5.0)
TOKEN_SENSITIVITY_TRANSLATION_M = 0.02
TOKEN_SENSITIVITY_SEED = TRAIN_PAIR_SEED + 910000

token_sensitivity_indices = np.linspace(0, len(train_dataset) - 1, TOKEN_SENSITIVITY_NUM_WINDOWS, dtype=int)
token_sensitivity_base_window = collate_windows([train_dataset[int(index)] for index in token_sensitivity_indices])
token_sensitivity_base_window = window_batch_to(token_sensitivity_base_window, DEVICE, dtype=MODEL_DTYPE)

token_sensitivity_generator = make_augmentation_generator(TOKEN_SENSITIVITY_SEED)
token_sensitivity_window = context_augmentation_pipeline(token_sensitivity_base_window, generator=token_sensitivity_generator).augmented_window
token_sensitivity_truth = token_sensitivity_window.current_calibration

component_names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z")

print("=" * 100)
print("SPATIAL CALIBRATION -> TOKEN SENSITIVITY")
print("=" * 100)
print(f"{'calibration':12s} {'component':10s} {'mean |Δx|':>14s} {'RMS Δx':>14s} {'max |Δx|':>14s} {'RMS/signal':>14s}")

for calibration_id, calibration_key in enumerate(CALIBRATION_KEYS):
    batch_size = token_sensitivity_truth[calibration_key].transform.shape[0]

    for component_index, component_name in enumerate(component_names):
        delta_xi_by_key = {key: torch.zeros(batch_size, 6, device=DEVICE, dtype=MODEL_DTYPE) for key in CALIBRATION_KEYS}
        delta_tau_by_key = {key: torch.zeros(batch_size, 1, device=DEVICE, dtype=MODEL_DTYPE) for key in CALIBRATION_KEYS}
        delta_xi_by_key[calibration_key][:, component_index] = TOKEN_SENSITIVITY_ROTATION_RAD if component_index < 3 else TOKEN_SENSITIVITY_TRANSLATION_M

        positive_calibration = apply_prior_perturbation(token_sensitivity_truth, delta_xi_by_key, delta_tau_by_key)
        negative_calibration = apply_prior_perturbation(token_sensitivity_truth, {key: -value for key, value in delta_xi_by_key.items()}, delta_tau_by_key)

        positive_tokens = tokenizer(geometry_processor(token_sensitivity_window.streams, token_sensitivity_window.metadata, positive_calibration), None)
        negative_tokens = tokenizer(geometry_processor(token_sensitivity_window.streams, token_sensitivity_window.metadata, negative_calibration), None)

        torch.testing.assert_close(positive_tokens.token_mask, negative_tokens.token_mask)
        torch.testing.assert_close(positive_tokens.calibration_ids, negative_tokens.calibration_ids)

        calibration_mask = positive_tokens.token_mask & (positive_tokens.calibration_ids == calibration_id)
        positive_measurements = positive_tokens.x[..., :MEASUREMENT_DIM][calibration_mask]
        negative_measurements = negative_tokens.x[..., :MEASUREMENT_DIM][calibration_mask]
        difference = positive_measurements - negative_measurements

        mean_abs = difference.abs().mean().item()
        rms = difference.square().mean().sqrt().item()
        maximum = difference.abs().max().item()
        signal_rms = (0.5 * (positive_measurements.square() + negative_measurements.square())).mean().sqrt().item()
        relative_rms = rms / max(signal_rms, 1e-12)

        print(f"{calibration_key:12s} {component_name:10s} {mean_abs:14.6e} {rms:14.6e} {maximum:14.6e} {relative_rms:14.6e}")

del token_sensitivity_base_window
del token_sensitivity_window
torch.cuda.empty_cache()

## 14. Paired epoch runner


In [ ]:
def run_paired_epoch(loader, *, training: bool, pair_generator: torch.Generator, global_step: int):
    model.train(training)

    metric_sums = {}
    num_samples = 0
    start_time = time.perf_counter()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    grad_context = torch.enable_grad() if training else torch.no_grad()

    with grad_context:
        for batch_index, base_window in enumerate(loader):
            base_window = window_batch_to(base_window, DEVICE, dtype=MODEL_DTYPE)
            positive_window, negative_window, _, _ = make_paired_windows(base_window, generator=pair_generator)

            if training:
                optimizer.zero_grad(set_to_none=True)

            positive_result = window_step(positive_window)
            negative_result = window_step(negative_window)
            loss, metrics = compute_paired_objective(positive_window, positive_result.model_output, negative_window, negative_result.model_output)

            if training:
                loss.backward()
                if GRADIENT_CLIP_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                optimizer.step()
                global_step += 1

            batch_size = next(iter(positive_window.streams.values())).values.shape[0]
            accumulate_metrics(metric_sums, metrics, batch_size)
            num_samples += batch_size

            if training and PRINT_EVERY_BATCHES > 0 and (batch_index + 1) % PRINT_EVERY_BATCHES == 0:
                print(f"batch {batch_index + 1:5d}/{len(loader):5d} loss={loss.item():.6f}")

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    elapsed_s = time.perf_counter() - start_time
    epoch_metrics = normalize_metrics(metric_sums, num_samples)
    epoch_metrics["epoch_time_s"] = elapsed_s
    epoch_metrics["samples_per_s"] = num_samples / elapsed_s

    return epoch_metrics, global_step


## 15. Checkpoint paths and experiment snapshot


In [ ]:
RUN_DIR = ROOT / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = RUN_DIR / "best.pt"
LAST_CHECKPOINT_PATH = RUN_DIR / "last.pt"
METRICS_PATH = RUN_DIR / "metrics.jsonl"

EXPERIMENT_SNAPSHOT = {
    "run_name": RUN_NAME,
    "calibration_keys": CALIBRATION_KEYS,
    "architecture": "PooledCalibrationModel",
    "paired_prior_training": True,
    "pair_loss_weight": PAIR_LOSS_WEIGHT,
    "pair_rotation_max_rad": PAIR_ROTATION_MAX_RAD,
    "pair_translation_max_m": PAIR_TRANSLATION_MAX_M,
    "pair_time_offset_max_s": PAIR_TIME_OFFSET_MAX_S,
    "frame_randomization": False,
    "true_calibration_augmentation": True,
    "measurement_noise": False,
    "numeric_calibration_context_in_pooler": True,
    "numeric_calibration_context_in_head": False,
    "observability_input": False,
}

train_pair_generator = make_augmentation_generator(TRAIN_PAIR_SEED)


## 16. Optional resume


In [ ]:
start_epoch = 0
global_step = 0
best_validation_loss = float("inf")

if RESUME_CHECKPOINT is not None:
    checkpoint = load_training_checkpoint(
        RESUME_CHECKPOINT,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        map_location=DEVICE,
        restore_rng_state=True,
        generators={"train_shuffle": train_shuffle_generator, "train_pair": train_pair_generator},
    )
    start_epoch = int(checkpoint.get("epoch", 0))
    global_step = int(checkpoint.get("global_step", 0))
    best_validation_loss = float(checkpoint.get("extra_state", {}).get("best_validation_loss", float("inf")))

    print("Resumed from:", RESUME_CHECKPOINT)
    print("start epoch:", start_epoch)
    print("global step:", global_step)


## 17. Main paired training loop


In [ ]:
torch.cuda.empty_cache()

In [ ]:
history = []
RUN_FULL_TRAINING = True
if RUN_FULL_TRAINING:
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    with MetricsLogger(METRICS_PATH, append=RESUME_CHECKPOINT is not None) as logger:
        if RESUME_CHECKPOINT is None:
            logger.log_config(model_config, name="model_config")
            logger.log_config(context_augmentation_config, name="context_augmentation_config")
            logger.log_config(pair_prior_config, name="pair_prior_config")
            logger.log_config(
                {
                    "device": str(DEVICE),
                    "batch_size": BATCH_SIZE,
                    "window_duration_s": WINDOW_DURATION_S,
                    "window_stride_s": WINDOW_STRIDE_S,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "train_sequences": TRAIN_NAMES,
                    "validation_sequences": VAL_NAMES,
                    "test_sequences": TEST_NAMES,
                    **EXPERIMENT_SNAPSHOT,
                },
                name="experiment",
            )

        for epoch in range(start_epoch, NUM_EPOCHS):
            print()
            print("=" * 80)
            print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")

            train_metrics, global_step = run_paired_epoch(train_loader, training=True, pair_generator=train_pair_generator, global_step=global_step)

            # Recreate validation generator from the same seed every epoch so validation pairs stay fixed.
            val_pair_generator = make_augmentation_generator(VALIDATION_PAIR_SEED)
            val_metrics, _ = run_paired_epoch(val_loader, training=False, pair_generator=val_pair_generator, global_step=global_step)

            scheduler.step(val_metrics["loss_total"])
            learning_rate = optimizer.param_groups[0]["lr"]

            row = {
                "epoch": epoch + 1,
                "global_step": global_step,
                "learning_rate": learning_rate,
                **{f"train/{key}": value for key, value in train_metrics.items()},
                **{f"val/{key}": value for key, value in val_metrics.items()},
            }
            history.append(row)

            logger.log_metrics({**train_metrics, "learning_rate": learning_rate}, split="train", epoch=epoch + 1, step=global_step)
            logger.log_metrics(val_metrics, split="validation", epoch=epoch + 1, step=global_step)

            print(
                f"train total={train_metrics['loss_total']:.6f} regular={train_metrics['loss_regular']:.6f} pair={train_metrics['loss_pair']:.6f} | "
                f"val total={val_metrics['loss_total']:.6f} regular={val_metrics['loss_regular']:.6f} pair={val_metrics['loss_pair']:.6f} | "
                f"lr={learning_rate:.3e}"
            )

            if val_metrics["loss_total"] < best_validation_loss:
                best_validation_loss = val_metrics["loss_total"]
                save_training_checkpoint(
                    BEST_CHECKPOINT_PATH,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch + 1,
                    global_step=global_step,
                    metrics=val_metrics,
                    generators={"train_shuffle": train_shuffle_generator, "train_pair": train_pair_generator},
                    extra_state={"best_validation_loss": best_validation_loss},
                )
                print("Saved new best checkpoint.")

            if (epoch + 1) % CHECKPOINT_EVERY_EPOCHS == 0 or epoch + 1 == NUM_EPOCHS:
                save_training_checkpoint(
                    LAST_CHECKPOINT_PATH,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch + 1,
                    global_step=global_step,
                    metrics=val_metrics,
                    generators={"train_shuffle": train_shuffle_generator, "train_pair": train_pair_generator},
                    extra_state={"best_validation_loss": best_validation_loss},
                )
                print("Saved last checkpoint.")
else:
    print("Full training skipped. Fix the paired stochastic learnability gate before spending 200 epochs.")


## 18. Training curves


In [ ]:
if history:
    epochs = [row["epoch"] for row in history]

    plt.figure(figsize=(12, 5))
    plt.plot(epochs, [row["train/loss_total"] for row in history], label="train total")
    plt.plot(epochs, [row["val/loss_total"] for row in history], label="validation total")
    plt.plot(epochs, [row["train/loss_regular"] for row in history], label="train regular", alpha=0.7)
    plt.plot(epochs, [row["train/loss_pair"] for row in history], label="train pair", alpha=0.7)
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Paired pooled calibration training")
    plt.grid(True)
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(epochs, [row["learning_rate"] for row in history])
    plt.xlabel("epoch")
    plt.ylabel("learning rate")
    plt.title("Learning-rate schedule")
    plt.grid(True)
    plt.show()
torch.cuda.empty_cache()

## 19. Evaluate the best checkpoint


In [ ]:
if BEST_CHECKPOINT_PATH.exists():
    best_checkpoint = load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
    test_pair_generator = make_augmentation_generator(TEST_PAIR_SEED)
    test_metrics, _ = run_paired_epoch(test_loader, training=False, pair_generator=test_pair_generator, global_step=global_step)

    print("Best checkpoint epoch:", best_checkpoint.get("epoch"))
    print("Test metrics:")
    for key in sorted(test_metrics):
        print(f"  {key}: {test_metrics[key]:.6f}")
else:
    print("No best checkpoint exists yet.")


## 20. Select held-out windows for detailed visualization


In [ ]:
VIS_SEQUENCE_NAME = "Urban38"
VIS_START_WINDOW = 0
VIS_NUM_WINDOWS = 100
VIS_BATCH_SIZE = 16

sequence_indices = [index for index in range(len(test_dataset)) if test_dataset.sequence_name_for_window(index) == VIS_SEQUENCE_NAME]

if not sequence_indices:
    raise ValueError(f"Sequence {VIS_SEQUENCE_NAME!r} is not present in test_dataset.")

selected_indices = sequence_indices[VIS_START_WINDOW : VIS_START_WINDOW + VIS_NUM_WINDOWS]

print("Sequence:", VIS_SEQUENCE_NAME)
print("Selected windows:", len(selected_indices))
print("Dataset indices:", selected_indices[0], "...", selected_indices[-1])


## 21. Collect deterministic paired predictions


In [ ]:
def collect_paired_predictions(dataset, indices, *, batch_size, pair_seed):
    model.eval()
    pair_generator = make_augmentation_generator(pair_seed)

    collected = {
        calibration_key: {
            "positive_true_xi": [],
            "positive_predicted_xi": [],
            "positive_true_tau": [],
            "positive_predicted_tau": [],
            "negative_true_xi": [],
            "negative_predicted_xi": [],
            "negative_true_tau": [],
            "negative_predicted_tau": [],
        }
        for calibration_key in CALIBRATION_KEYS
    }

    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            base_window = collate_windows([dataset[index] for index in batch_indices])
            base_window = window_batch_to(base_window, DEVICE, dtype=MODEL_DTYPE)

            positive_window, negative_window, _, _ = make_paired_windows(base_window, generator=pair_generator)
            positive_result = window_step(positive_window)
            negative_result = window_step(negative_window)

            for calibration_key in CALIBRATION_KEYS:
                positive_true_xi, positive_true_tau = correction_target(positive_window, calibration_key)
                negative_true_xi, negative_true_tau = correction_target(negative_window, calibration_key)

                positive_prediction = positive_result.model_output.predictions[calibration_key]
                negative_prediction = negative_result.model_output.predictions[calibration_key]

                collected[calibration_key]["positive_true_xi"].append(positive_true_xi.detach().cpu())
                collected[calibration_key]["positive_predicted_xi"].append(positive_prediction.delta_xi.detach().cpu())
                collected[calibration_key]["positive_true_tau"].append(positive_true_tau.detach().cpu())
                collected[calibration_key]["positive_predicted_tau"].append(positive_prediction.delta_tau.detach().cpu())

                collected[calibration_key]["negative_true_xi"].append(negative_true_xi.detach().cpu())
                collected[calibration_key]["negative_predicted_xi"].append(negative_prediction.delta_xi.detach().cpu())
                collected[calibration_key]["negative_true_tau"].append(negative_true_tau.detach().cpu())
                collected[calibration_key]["negative_predicted_tau"].append(negative_prediction.delta_tau.detach().cpu())

    for calibration_key in CALIBRATION_KEYS:
        for field_name in collected[calibration_key]:
            collected[calibration_key][field_name] = torch.cat(collected[calibration_key][field_name], dim=0).numpy()

    return collected


if BEST_CHECKPOINT_PATH.exists():
    load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)

visualization_results = collect_paired_predictions(test_dataset, selected_indices, batch_size=VIS_BATCH_SIZE, pair_seed=VISUALIZATION_PAIR_SEED)


## 22. True vs predicted correction over windows


In [ ]:
def physical_correction_values(xi, tau):
    return np.column_stack([np.rad2deg(xi[:, :3]), 100.0 * xi[:, 3:], 1000.0 * tau.reshape(-1)])


def plot_paired_true_vs_predicted(results, calibration_key):
    result = results[calibration_key]

    positive_true = physical_correction_values(result["positive_true_xi"], result["positive_true_tau"])
    positive_predicted = physical_correction_values(result["positive_predicted_xi"], result["positive_predicted_tau"])
    negative_true = physical_correction_values(result["negative_true_xi"], result["negative_true_tau"])
    negative_predicted = physical_correction_values(result["negative_predicted_xi"], result["negative_predicted_tau"])

    component_names = (r"$\phi_x$", r"$\phi_y$", r"$\phi_z$", r"$\rho_x$", r"$\rho_y$", r"$\rho_z$", r"$\tau$")
    units = ("deg", "deg", "deg", "cm", "cm", "cm", "ms")
    x = np.arange(len(positive_true))

    fig, axes = plt.subplots(4, 2, figsize=(15, 13), sharex=True)
    axes = axes.ravel()

    for component_index in range(7):
        ax = axes[component_index]
        ax.plot(x, positive_true[:, component_index], label="true +delta")
        ax.plot(x, positive_predicted[:, component_index], label="pred +delta")
        ax.plot(x, negative_true[:, component_index], label="true -delta", alpha=0.65)
        ax.plot(x, negative_predicted[:, component_index], label="pred -delta", alpha=0.65)
        ax.set_ylabel(f"{component_names[component_index]} [{units[component_index]}]")
        ax.grid(True)

        if component_index == 0:
            ax.legend(ncol=2)

    axes[6].set_xlabel("window")
    axes[7].axis("off")
    fig.suptitle(f"{calibration_key}: paired true vs predicted correction")
    fig.tight_layout()
    plt.show()


for calibration_key in CALIBRATION_KEYS:
    plot_paired_true_vs_predicted(visualization_results, calibration_key)


## 23. Combined true-vs-predicted scatter


In [ ]:
def plot_combined_true_vs_predicted_scatter(results, calibration_key):
    result = results[calibration_key]

    positive_true = physical_correction_values(result["positive_true_xi"], result["positive_true_tau"])
    positive_predicted = physical_correction_values(result["positive_predicted_xi"], result["positive_predicted_tau"])
    negative_true = physical_correction_values(result["negative_true_xi"], result["negative_true_tau"])
    negative_predicted = physical_correction_values(result["negative_predicted_xi"], result["negative_predicted_tau"])

    truth = np.concatenate([positive_true, negative_true], axis=0)
    prediction = np.concatenate([positive_predicted, negative_predicted], axis=0)

    component_names = (r"$\phi_x$", r"$\phi_y$", r"$\phi_z$", r"$\rho_x$", r"$\rho_y$", r"$\rho_z$", r"$\tau$")
    units = ("deg", "deg", "deg", "cm", "cm", "cm", "ms")

    fig, axes = plt.subplots(4, 2, figsize=(13, 14))
    axes = axes.ravel()

    for component_index in range(7):
        ax = axes[component_index]
        component_truth = truth[:, component_index]
        component_prediction = prediction[:, component_index]

        minimum = min(component_truth.min(), component_prediction.min())
        maximum = max(component_truth.max(), component_prediction.max())
        correlation = np.corrcoef(component_truth, component_prediction)[0, 1] if np.std(component_truth) > 0.0 and np.std(component_prediction) > 0.0 else np.nan
        mae = np.mean(np.abs(component_prediction - component_truth))

        ax.scatter(component_truth, component_prediction, s=18, alpha=0.7)
        ax.plot([minimum, maximum], [minimum, maximum], linestyle="--")
        ax.set_xlabel(f"true [{units[component_index]}]")
        ax.set_ylabel(f"predicted [{units[component_index]}]")
        ax.set_title(f"{component_names[component_index]}   corr={correlation:+.3f}, MAE={mae:.3f} {units[component_index]}")
        ax.grid(True)

    axes[7].axis("off")
    fig.suptitle(f"{calibration_key}: combined +/- paired predictions")
    fig.tight_layout()
    plt.show()


for calibration_key in CALIBRATION_KEYS:
    plot_combined_true_vs_predicted_scatter(visualization_results, calibration_key)


## 24. Pair-difference scatter

This is the most important anti-collapse visualization. A constant predictor produces approximately zero pair difference and therefore a horizontal cloud. A useful paired model should approach the identity diagonal.


In [ ]:
def plot_pair_difference_scatter(results, calibration_key):
    result = results[calibration_key]

    positive_true = physical_correction_values(result["positive_true_xi"], result["positive_true_tau"])
    positive_predicted = physical_correction_values(result["positive_predicted_xi"], result["positive_predicted_tau"])
    negative_true = physical_correction_values(result["negative_true_xi"], result["negative_true_tau"])
    negative_predicted = physical_correction_values(result["negative_predicted_xi"], result["negative_predicted_tau"])

    target_difference = positive_true - negative_true
    predicted_difference = positive_predicted - negative_predicted

    component_names = (r"$\phi_x$", r"$\phi_y$", r"$\phi_z$", r"$\rho_x$", r"$\rho_y$", r"$\rho_z$", r"$\tau$")
    units = ("deg", "deg", "deg", "cm", "cm", "cm", "ms")

    fig, axes = plt.subplots(4, 2, figsize=(13, 14))
    axes = axes.ravel()

    for component_index in range(7):
        ax = axes[component_index]
        truth = target_difference[:, component_index]
        prediction = predicted_difference[:, component_index]

        minimum = min(truth.min(), prediction.min())
        maximum = max(truth.max(), prediction.max())
        correlation = np.corrcoef(truth, prediction)[0, 1] if np.std(truth) > 0.0 and np.std(prediction) > 0.0 else np.nan

        ax.scatter(truth, prediction, s=18, alpha=0.7)
        ax.plot([minimum, maximum], [minimum, maximum], linestyle="--")
        ax.set_xlabel(f"target pair difference [{units[component_index]}]")
        ax.set_ylabel(f"predicted pair difference [{units[component_index]}]")
        ax.set_title(f"{component_names[component_index]}   corr={correlation:+.3f}")
        ax.grid(True)

    axes[7].axis("off")
    fig.suptitle(f"{calibration_key}: +delta/-delta separation")
    fig.tight_layout()
    plt.show()


for calibration_key in CALIBRATION_KEYS:
    plot_pair_difference_scatter(visualization_results, calibration_key)


## 25. Numerical paired-prediction summary


In [ ]:
def print_paired_prediction_summary(results, calibration_key):
    result = results[calibration_key]

    positive_true = physical_correction_values(result["positive_true_xi"], result["positive_true_tau"])
    positive_predicted = physical_correction_values(result["positive_predicted_xi"], result["positive_predicted_tau"])
    negative_true = physical_correction_values(result["negative_true_xi"], result["negative_true_tau"])
    negative_predicted = physical_correction_values(result["negative_predicted_xi"], result["negative_predicted_tau"])

    truth = np.concatenate([positive_true, negative_true], axis=0)
    prediction = np.concatenate([positive_predicted, negative_predicted], axis=0)
    target_difference = positive_true - negative_true
    predicted_difference = positive_predicted - negative_predicted

    component_names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z", "tau")
    units = ("deg", "deg", "deg", "cm", "cm", "cm", "ms")

    print(calibration_key)
    print(f"{'component':8s} {'MAE':>12s} {'corr':>10s} {'pred std':>12s} {'pair corr':>12s} {'pair MAE':>12s}")

    for component_index, component_name in enumerate(component_names):
        component_truth = truth[:, component_index]
        component_prediction = prediction[:, component_index]
        difference_truth = target_difference[:, component_index]
        difference_prediction = predicted_difference[:, component_index]

        mae = np.mean(np.abs(component_prediction - component_truth))
        correlation = np.corrcoef(component_truth, component_prediction)[0, 1] if np.std(component_truth) > 0.0 and np.std(component_prediction) > 0.0 else np.nan
        pair_correlation = np.corrcoef(difference_truth, difference_prediction)[0, 1] if np.std(difference_truth) > 0.0 and np.std(difference_prediction) > 0.0 else np.nan
        pair_mae = np.mean(np.abs(difference_prediction - difference_truth))

        print(
            f"{component_name:8s} "
            f"{mae:9.4f} {units[component_index]:3s} "
            f"{correlation:+10.4f} "
            f"{np.std(component_prediction):12.4f} "
            f"{pair_correlation:+12.4f} "
            f"{pair_mae:9.4f} {units[component_index]:3s}"
        )

    print()


for calibration_key in CALIBRATION_KEYS:
    print_paired_prediction_summary(visualization_results, calibration_key)


## 26. Exact-truth fixed-point diagnostic


In [ ]:
FIXED_POINT_NUM_WINDOWS = min(100, len(sequence_indices))
fixed_point_positions = np.unique(np.linspace(0, len(sequence_indices) - 1, FIXED_POINT_NUM_WINDOWS).round().astype(int))
fixed_point_indices = np.asarray(sequence_indices)[fixed_point_positions]

fixed_point_results = {calibration_key: {"delta_xi": [], "delta_tau": []} for calibration_key in CALIBRATION_KEYS}

model.eval()

with torch.no_grad():
    for dataset_index in fixed_point_indices:
        window = window_batch_to(collate_windows([test_dataset[int(dataset_index)]]), DEVICE, dtype=MODEL_DTYPE)
        exact_true_calibration = {calibration_key: CalibrationState(transform=state.transform.clone(), time_offset=state.time_offset.clone()) for calibration_key, state in window.current_calibration.items()}
        result = window_step(window, calibration=exact_true_calibration)

        for calibration_key, prediction in result.model_output.predictions.items():
            fixed_point_results[calibration_key]["delta_xi"].append(prediction.delta_xi[0].detach().cpu())
            fixed_point_results[calibration_key]["delta_tau"].append(prediction.delta_tau[0, 0].detach().cpu())

for calibration_key in CALIBRATION_KEYS:
    fixed_point_results[calibration_key]["delta_xi"] = torch.stack(fixed_point_results[calibration_key]["delta_xi"]).numpy()
    fixed_point_results[calibration_key]["delta_tau"] = torch.stack(fixed_point_results[calibration_key]["delta_tau"]).numpy()

component_names = ("phi_x", "phi_y", "phi_z", "rho_x", "rho_y", "rho_z", "tau")

print("=" * 120)
print("EXACT-TRUTH FIXED-POINT DIAGNOSTIC")
print("=" * 120)

for calibration_key in CALIBRATION_KEYS:
    result = fixed_point_results[calibration_key]
    prediction = np.concatenate([result["delta_xi"], result["delta_tau"][:, None]], axis=1)

    print(calibration_key)
    print(f"{'component':8s} {'mean':>16s} {'mean |x|':>16s} {'RMSE':>16s} {'std':>16s} {'range':>16s}")

    for component_index, component_name in enumerate(component_names):
        values = prediction[:, component_index]

        if component_index < 3:
            values, unit = np.rad2deg(values), "deg"
        elif component_index < 6:
            values, unit = 100.0 * values, "cm"
        else:
            values, unit = 1000.0 * values, "ms"

        mean = np.mean(values)
        mean_abs = np.mean(np.abs(values))
        rmse = np.sqrt(np.mean(values ** 2))
        std = np.std(values)
        value_range = np.max(values) - np.min(values)

        print(f"{component_name:8s} {mean:+12.6f} {unit:3s} {mean_abs:12.6f} {unit:3s} {rmse:12.6f} {unit:3s} {std:12.6f} {unit:3s} {value_range:12.6f} {unit:3s}")

    print()


## 27. Interpretation gate

Use the paired stochastic small-set diagnostic before trusting a full run.

A healthy paired model should satisfy all of the following:

1. the fixed paired overfit test strongly converges;
2. the stochastic paired small-set evaluation falls substantially;
3. held-out prediction standard deviations are nonzero;
4. true-vs-predicted scatter moves toward the identity line;
5. pair-difference scatter moves toward the identity line;
6. exact-truth inputs still produce corrections close to zero.

If the paired stochastic small-set test still collapses, do not compensate by merely increasing `PAIR_LOSS_WEIGHT` indefinitely. The next architectural experiment should expose the current calibration numerically to the pooled prediction path, because the present architecture then appears unable to recover enough calibration dependence indirectly from geometry-processed measurements.


# Shuffle diagnostics of prior-only training

In [ ]:
# Measurement-shuffle diagnostic.
# Keep every calibration prior and supervision target attached to its original batch item,
# but replace the complete measurement window by another item's measurement window.
# If performance barely changes, the model can solve the task mostly from calibration context alone.
# If performance degrades substantially, the model is genuinely using the sensor measurements.

from obscalib.data.structures import SensorStreamBatch, WindowBatch

MEASUREMENT_SHUFFLE_NUM_WINDOWS = min(BATCH_SIZE, 16)
MEASUREMENT_SHUFFLE_PAIR_SEED = TEST_PAIR_SEED + 700000
MEASUREMENT_SHUFFLE_PERMUTATION_SEED = TEST_PAIR_SEED + 800000

measurement_shuffle_indices = np.linspace(0, len(test_dataset) - 1, MEASUREMENT_SHUFFLE_NUM_WINDOWS, dtype=int)
measurement_shuffle_base_window = collate_windows([test_dataset[int(index)] for index in measurement_shuffle_indices])
measurement_shuffle_base_window = window_batch_to(measurement_shuffle_base_window, DEVICE, dtype=MODEL_DTYPE)

measurement_shuffle_pair_generator = make_augmentation_generator(MEASUREMENT_SHUFFLE_PAIR_SEED)

measurement_shuffle_positive_window, measurement_shuffle_negative_window, _, _ = make_paired_windows(
    measurement_shuffle_base_window,
    generator=measurement_shuffle_pair_generator,
)

batch_size = next(iter(measurement_shuffle_positive_window.streams.values())).values.shape[0]

if batch_size < 2:
    raise ValueError("Measurement-shuffle diagnostic requires at least two batch items.")

# Build a deterministic random derangement so no batch item keeps its own measurements.
permutation_rng = np.random.default_rng(MEASUREMENT_SHUFFLE_PERMUTATION_SEED)
identity_permutation = np.arange(batch_size)

while True:
    measurement_permutation_numpy = permutation_rng.permutation(batch_size)

    if np.all(measurement_permutation_numpy != identity_permutation):
        break

measurement_permutation = torch.as_tensor(measurement_permutation_numpy, dtype=torch.long)


def shuffle_window_measurements(window: WindowBatch, permutation: torch.Tensor) -> WindowBatch:
    shuffled_streams = {}

    for stream_key, stream in window.streams.items():
        stream_permutation = permutation.to(stream.values.device)

        shuffled_streams[stream_key] = SensorStreamBatch(
            values=stream.values[stream_permutation],
            timestamps=stream.timestamps[stream_permutation],
            sample_mask=stream.sample_mask[stream_permutation],
            interval_start_timestamps=None if stream.interval_start_timestamps is None else stream.interval_start_timestamps[stream_permutation],
        )

    return WindowBatch(
        streams=shuffled_streams,
        current_calibration=dict(window.current_calibration),
        metadata=dict(window.metadata),
        targets=None if window.targets is None else dict(window.targets),
        dataset_indices=window.dataset_indices,
    )


measurement_shuffle_positive_shuffled_window = shuffle_window_measurements(measurement_shuffle_positive_window, measurement_permutation)
measurement_shuffle_negative_shuffled_window = shuffle_window_measurements(measurement_shuffle_negative_window, measurement_permutation)


def evaluate_measurement_shuffle_pair(positive_window: WindowBatch, negative_window: WindowBatch):
    with torch.inference_mode():
        positive_result = window_step(positive_window)
        positive_output = positive_result.model_output
        del positive_result

        negative_result = window_step(negative_window)
        negative_output = negative_result.model_output
        del negative_result

        loss, metrics = compute_paired_objective(
            positive_window,
            positive_output,
            negative_window,
            negative_output,
        )

        prediction_summary = {}

        for calibration_key in CALIBRATION_KEYS:
            prediction_summary[calibration_key] = {
                "positive_xi": positive_output.predictions[calibration_key].delta_xi.detach().cpu(),
                "negative_xi": negative_output.predictions[calibration_key].delta_xi.detach().cpu(),
                "positive_tau": positive_output.predictions[calibration_key].delta_tau.detach().cpu(),
                "negative_tau": negative_output.predictions[calibration_key].delta_tau.detach().cpu(),
            }

        loss_value = float(loss.detach().cpu())
        metric_values = {key: float(value.detach().cpu()) if torch.is_tensor(value) else float(value) for key, value in metrics.items()}

        del positive_output
        del negative_output
        del loss

    return loss_value, metric_values, prediction_summary


previous_training_mode = model.training
model.eval()

try:
    normal_total, normal_metrics, normal_predictions = evaluate_measurement_shuffle_pair(
        measurement_shuffle_positive_window,
        measurement_shuffle_negative_window,
    )

    shuffled_total, shuffled_metrics, shuffled_predictions = evaluate_measurement_shuffle_pair(
        measurement_shuffle_positive_shuffled_window,
        measurement_shuffle_negative_shuffled_window,
    )

finally:
    model.train(previous_training_mode)


print("=" * 110)
print("MEASUREMENT-SHUFFLE DEPENDENCE DIAGNOSTIC")
print("=" * 110)
print("windows:", batch_size)
print("measurement permutation:", measurement_permutation_numpy.tolist())
print()
print("Calibration priors: unchanged")
print("Supervision targets: unchanged")
print("Measurements: complete physical windows reassigned between batch items")
print()
print(f"{'condition':14s} {'total':>12s} {'regular':>12s} {'pair':>12s}")
print(f"{'normal':14s} {normal_total:12.6f} {normal_metrics['loss_regular']:12.6f} {normal_metrics['loss_pair']:12.6f}")
print(f"{'shuffled':14s} {shuffled_total:12.6f} {shuffled_metrics['loss_regular']:12.6f} {shuffled_metrics['loss_pair']:12.6f}")
print()

total_ratio = shuffled_total / max(normal_total, 1e-12)
regular_ratio = shuffled_metrics["loss_regular"] / max(normal_metrics["loss_regular"], 1e-12)
pair_ratio = shuffled_metrics["loss_pair"] / max(normal_metrics["loss_pair"], 1e-12)

print(f"shuffled / normal total loss:   {total_ratio:.3f}x")
print(f"shuffled / normal regular loss: {regular_ratio:.3f}x")
print(f"shuffled / normal pair loss:    {pair_ratio:.3f}x")
print()

for calibration_key in CALIBRATION_KEYS:
    normal = normal_predictions[calibration_key]
    shuffled = shuffled_predictions[calibration_key]

    positive_xi_change = torch.linalg.vector_norm(shuffled["positive_xi"] - normal["positive_xi"], dim=-1).mean().item()
    negative_xi_change = torch.linalg.vector_norm(shuffled["negative_xi"] - normal["negative_xi"], dim=-1).mean().item()
    positive_tau_change = torch.abs(shuffled["positive_tau"] - normal["positive_tau"]).mean().item()
    negative_tau_change = torch.abs(shuffled["negative_tau"] - normal["negative_tau"]).mean().item()

    print(calibration_key)
    print(f"  mean prediction change after measurement shuffle, + branch: xi={positive_xi_change:.6e}, tau={positive_tau_change:.6e}")
    print(f"  mean prediction change after measurement shuffle, - branch: xi={negative_xi_change:.6e}, tau={negative_tau_change:.6e}")
    print()

print("Interpretation:")
print("  Large loss increase and/or large prediction changes -> the model uses measurement content.")
print("  Nearly unchanged losses and predictions             -> the model is relying mostly on calibration context.")

del measurement_shuffle_base_window
del measurement_shuffle_positive_window
del measurement_shuffle_negative_window
del measurement_shuffle_positive_shuffled_window
del measurement_shuffle_negative_shuffled_window
del normal_predictions
del shuffled_predictions

torch.cuda.empty_cache()